# License:

## Connecting blue-green infrastructure elements to reduce combined sewer overflows

Cavadini, Rodriguez, Cook 

EAWAG, 2024

Citation
Cavadini et al., 2024 - https://doi.org/10.1016/j.jenvman.2024.121465

Copyright ©, 2024, Eawag

Contact: lauren.cook@eawag.ch, giovanbattista.cavadini@eawag.ch

Permission is hereby granted, free of charge, to any person obtaining
a copy of this software and associated documentation files (the
"Software"), to deal in the Software without restriction, including
without limitation the rights to use, copy, modify, merge, publish,
distribute, sublicense, and/or sell copies of the Software, and to
permit persons to whom the Software is furnished to do so, subject to
the following conditions:

The above copyright notice and this permission notice shall be
included in all copies or substantial portions of the Software.

THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND,
EXPRESS OR IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF
MERCHANTABILITY, FITNESS FOR A PARTICULAR PURPOSE AND
NONINFRINGEMENT. IN NO EVENT SHALL THE AUTHORS OR COPYRIGHT HOLDERS BE
LIABLE FOR ANY CLAIM, DAMAGES OR OTHER LIABILITY, WHETHER IN AN ACTION
OF CONTRACT, TORT OR OTHERWISE, ARISING FROM, OUT OF OR IN CONNECTION
WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE SOFTWARE.

In [1]:
import pandas as pd
import numpy as np
import scipy.stats
import matplotlib.pyplot as plt
import datetime as dt
import seaborn as sns
import matplotlib
import os
from sklearn.metrics import r2_score
from pyproj import Transformer
import math

# Set paths and load data

In [2]:
#set path where the data are saved
#path = "" in this case, the files are in the same folder as the Jupyter notebook

#four input files required: total available surface of BGI types, COmbination of BGI types in the scenarios, percentage of implementation, characteristics of the subcatchments
tot_surface= pd.read_csv("suitable_bgi_area.csv", sep = ";") #excel file with the total surface availability
scenarios = pd.read_csv("scenarios_distribution.csv", sep = ";") #excel file with the scenarios
sub_parameters= pd.read_csv("subcatchment_characteristics.csv", sep = ";") #excel file with the parameters of the subcatchment

area_perc= [0.25, 0.50, 0.75, 1] #array of the different implementation percentage considered


#select the location of the original SWMM input file
SWMM_file = 'swmm_files/original_file/calibrated_surface_model_original_30min.inp'

## Set the rainfall return level

In [3]:
#source: https://www.meteoswiss.admin.ch/services-and-publications/applications/standard-period.html
#hourly intensity for two return periods (2 and 100 years)

rainfall_2y = 17.3 #mm/h
rainfall_100y = 56.5 #mm/h

## Show the input files

Dataframe with the scenarios combinations

In [4]:
scenarios=scenarios.set_index("BGI")
scenarios

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14
BGI,,,,,,,,,,,,,,,
BC,1,0,0,0,1,0,1,1,0,0,1,0,1,1,1
PP,0,1,0,0,1,1,0,0,1,0,1,1,0,1,1
GR,0,0,1,0,0,1,1,0,0,1,1,1,1,0,1
UP,0,0,0,1,0,0,0,1,1,1,0,1,1,1,1


Dataframe with the total suitable area for each BGI in each sub-catchment

In [5]:
tot_surface=tot_surface.set_index("Subcatchment MUID") 
tot_surface

,BC,PP,GR,UP
Subcatchment MUID,,,,
A0005RT,406.956,309.006209,0.000000,406.956
A0010RT,878.843,238.972042,0.000000,878.843
A0015RT,2217.753,829.347014,385.795364,2217.753
A0020RT,980.481,1413.579796,955.661840,980.481
A0025RT,5058.811,1560.841130,80.586888,5058.811
...,...,...,...,...
F0105RT,2740.664,961.670207,1061.231524,2740.664
F0110RT,1079.486,3065.636719,1987.859332,1079.486
F0120RT,1640.095,2443.224369,1609.574845,1640.095


Dataframe with sub-catchments' parameters

In [6]:
sub_parameters = sub_parameters.set_index("Name") 
sub_parameters

,X-Coordinate,Y-Coordinate,Outlet,Surface Elevation (m),Drainage Area (m2)
Name,,,,,
A0005RT,699519.250,248894.900,102a,536.16,384.8634
A0010RT,699471.938,248896.156,102c,536.06,568.9461
A0015RT,699710.250,249054.672,603,537.34,1735.8938
A0020RT,699725.500,249088.328,603,538.30,2730.9280
A0025RT,699803.500,249050.188,602,537.49,2982.1722
...,...,...,...,...,...
F0105RT,698201.750,249954.469,801c,520.69,2023.1796
F0110RT,698291.700,249848.063,P366,522.87,5053.9216
F0120RT,698291.563,249978.313,801g,525.78,4602.0491


In [7]:
#example on how the distance between the two points is calculated
# Define the two points in Swiss coordinates #A0005RT and A0010RT
x1 = 699519.250 
y1 = 248894.900
x2 = 699525.484
y2 = 248908.265

# Define the projection transformation
transformer = Transformer.from_crs('epsg:21781', 'epsg:4326')

# Convert Swiss coordinates to geographic coordinates
lat1, lon1 = transformer.transform(x1, y1)
lat2, lon2 = transformer.transform(x2, y2)

# Compute the great-circle distance between the two points
R = 6371  # Earth radius in kilometers
dlat = math.radians(lat2 - lat1)
dlon = math.radians(lon2 - lon1)
a = math.sin(dlat/2)**2 + math.cos(math.radians(lat1)) * math.cos(math.radians(lat2)) * math.sin(dlon/2)**2
c = 2 * math.atan2(math.sqrt(a), math.sqrt(1-a))
distance = 6371 * c *1000 #6371 Earth radius in kilometers
 
print(f"The distance between the two points is {distance:.2f} meters")


The distance between the two points is 14.74 meters


# Create the scenarios combinations

Function that needs as input the bgi surface, the bgi combinations and the implementation rates, and create the libraries with the relevant informations. It also sets the minimum area requirements. 


In [8]:
lib_scenarios = {}

def create_scenarios(tot_surf,scen,percentage):
    for i in range(len(scen.iloc[1,:])):
        for j in range(len(percentage)):
            name="sc_"+str(i)+"_"+str(percentage[j]) #create name for the library
            
            if scen.iloc[0,i]==1:
                name = name + "_BC"
            if scen.iloc[1,i]==1:
                name = name + "_PP"
            if scen.iloc[2,i]==1:
                name = name + "_GR"
            if scen.iloc[3,i]==1:
                name = name + "_UP"
            
            lib_scenarios[name]=percentage[j]*np.array(scen.iloc[:,i])*tot_surf #compute the combination
            
            #if both biocell and ponds are present, divide the implementation area by 2
            if scen.iloc[0,i]==1 and scen.iloc[3,i]==1:
            
                lib_scenarios[name]["BC"]=0.5*lib_scenarios[name]["BC"]
                lib_scenarios[name]["UP"]=0.5*lib_scenarios[name]["UP"]
                
            #ponds need a buffer 0f 7.5m, if the pervious area is not enough less, pond surface is set to 0
            lib_scenarios[name]["UP"][-lib_scenarios[name]["UP"]+tot_surface["BC"]<(7.5*2)**2-lib_scenarios[name]["UP"]]=0
                        
            #set to 0 if area is too small (see minimum requirements listed in the supplementary information)            
            lib_scenarios[name]["BC"][lib_scenarios[name]["BC"]<2]=0 #set to zero if biocell surface is less than 2 m2
            lib_scenarios[name]["PP"][lib_scenarios[name]["PP"]<2]=0 #set to zero if porous pavement surface is less than 2 m2
            lib_scenarios[name]["GR"][lib_scenarios[name]["GR"]<10]=0 #set to zero if green roof surface is less than 10 m2
            lib_scenarios[name]["UP"][lib_scenarios[name]["UP"]<18]=0 #set to zero if ponds surface is less than 18 m2
                
    return lib_scenarios

create_scenarios(tot_surface,scenarios,area_perc)

{'sc_0_0.25_BC':                            BC   PP   GR   UP
 Subcatchment MUID                           
 A0005RT             101.73900  0.0  0.0  0.0
 A0010RT             219.71075  0.0  0.0  0.0
 A0015RT             554.43825  0.0  0.0  0.0
 A0020RT             245.12025  0.0  0.0  0.0
 A0025RT            1264.70275  0.0  0.0  0.0
 ...                       ...  ...  ...  ...
 F0105RT             685.16600  0.0  0.0  0.0
 F0110RT             269.87150  0.0  0.0  0.0
 F0120RT             410.02375  0.0  0.0  0.0
 F0125RT             431.61275  0.0  0.0  0.0
 Kempttalstrasse     276.87300  0.0  0.0  0.0
 
 [246 rows x 4 columns],
 'sc_0_0.5_BC':                           BC   PP   GR   UP
 Subcatchment MUID                          
 A0005RT             203.4780  0.0  0.0  0.0
 A0010RT             439.4215  0.0  0.0  0.0
 A0015RT            1108.8765  0.0  0.0  0.0
 A0020RT             490.2405  0.0  0.0  0.0
 A0025RT            2529.4055  0.0  0.0  0.0
 ...                      ...

### Save the scenarios in excel files

In [9]:
#for i in lib_scenarios.keys():
#    name="scenario_"+str(i)+ ".xlsx"
#    lib_scenarios[i].to_excel(name)

## Create Swmm input file 

### Find relevant sections in the original SWMM input file

In [10]:
#find the locations in the input file that need to be modified 

with open(SWMM_file, 'r') as f:
    line_num = 0
    # Read each line in the file
    for line in f:
        # Increment the line number
        line_num += 1
        # Check if the line matches the line you are looking for
        if line.strip() == "[SUBCATCHMENTS]":
            # Print the line number and exit the loop
            start_sub = line_num
            print(f"Line found at index {line_num}")
        if line.strip() == "[SUBAREAS]":
            # Print the line number and exit the loop
            end_sub = line_num
            print(f"Line found at index {line_num}")
        if line.strip() == "[LID_USAGE]":
            # Print the line number and exit the loop
            start_LID = line_num
            print(f"Line found at index {line_num}")
        if line.strip() == "[AQUIFERS]":
            # Print the line number and exit the loop
            end_LID = line_num
            print(f"Line found at index {line_num}")
        if line.strip() == "[STORAGE]":
            # Print the line number and exit the loop
            start_storage = line_num
            print(f"Line found at index {line_num}")
        if line.strip() == "[CONDUITS]":
            # Print the line number and exit the loop
            end_storage = start_conduits = line_num
            print(f"Line found at index {line_num}")
        if line.strip() == "[PUMPS]":
            # Print the line number and exit the loop
            end_conduits = line_num
            print(f"Line found at index {line_num}")
        if line.strip() == "[XSECTIONS]":
            # Print the line number and exit the loop
            end_weirs = line_num
            print(f"Line found at index {line_num}")
        if line.strip() == "[LOSSES]":
            # Print the line number and exit the loop
            end_xsections = line_num
            print(f"Line found at index {line_num}") 
        if line.strip() == "[JUNCTIONS]":
            # Print the line number and exit the loop
            start_junctions = line_num
            print(f"Line found at index {line_num}")
        if line.strip() == "[OUTFALLS]":
            # Print the line number and exit the loop
            end_junctions = line_num
            print(f"Line found at index {line_num}")
        if line.strip() == "[COORDINATES]":
            # Print the line number and exit the loop
            start_coordinates = line_num
            print(f"Line found at index {line_num}") 
        if line.strip() == "[VERTICES]":
            # Print the line number and exit the loop
            end_coordinates = line_num
            print(f"Line found at index {line_num}")
            break
    else:
        # If the loop completes without finding the line, print a message
        print("Line not found")        

f.close()

# Open the file in read mode
with open(SWMM_file, 'r') as f:
    # Read the contents of the input file
    lines_input = f.readlines()
    coordinates_input = lines_input[start_coordinates+2:end_coordinates-2]
    

f.close()


coordinates_input_row = [row.split() for row in coordinates_input]


# Create the dataframe with the coordinates of the junctions
coord_j = pd.DataFrame(coordinates_input_row, columns=['Node', 'X-Coord', 'Y-Coord'])

coord_j = coord_j.rename(columns={'Node': 'Name'})

Line found at index 56
Line found at index 306
Line found at index 826
Line found at index 830
Line found at index 1085
Line found at index 1516
Line found at index 1528
Line found at index 1541
Line found at index 1976
Line found at index 2004
Line found at index 2451
Line found at index 3027
Line found at index 3471


### Create pandas table with the parameters of the subcatchment

In [11]:
subcatchment_pd=pd.Series(lines_input[start_sub+2:end_sub-2])
subcatchment_table = []

for i in range(len(subcatchment_pd)):
    subcatchment_table.append(subcatchment_pd[i].split())

pd_sub=pd.DataFrame(subcatchment_table, columns=['Name', 'Rain Gage', 'Outlet', 'Area', '%Imperv', 'Width', '%Slope', 'CurbLen'])

pd_sub['Drainage Area (m2)']=pd_sub['Area'].astype(float)*pd_sub['%Imperv'].astype(float)/100*10000 #compute drainage area

pd_sub

,Name,Rain Gage,Outlet,Area,%Imperv,Width,%Slope,CurbLen,Drainage Area (m2)
0,A0005RT,rain_gauge,102a,0.07919,48.6,28.1412,5.48,0,384.8634
1,A0010RT,rain_gauge,102c,0.14477,39.3,38.0482,4.93,0,568.9461
2,A0015RT,rain_gauge,603,0.39542,43.9,62.8821,6.88,0,1735.8938
3,A0020RT,rain_gauge,603,0.37105,73.6,60.9142,7.21,0,2730.9280
4,A0025RT,rain_gauge,602,0.80382,37.1,89.6562,6.86,0,2982.1722
...,...,...,...,...,...,...,...,...,...
241,F0105RT,rain_gauge,801c,0.63622,31.8,79.7634,9.59,0,2023.1796
242,F0110RT,rain_gauge,P366,0.61334,82.4,78.3158,9.35,0,5053.9216
243,F0120RT,rain_gauge,801g,0.62443,73.7,79.021,4.48,0,4602.0491
244,F0125RT,rain_gauge,801c,0.46993,63.2,68.5516,8.67,0,2969.9576


### Create pandas table with the parameters of the junctions (for the outflow pipe of the pond)

In [12]:
junction_pd=pd.Series(lines_input[start_junctions+2:end_junctions-2])
junction_table = []

for i in range(len(junction_pd)):
    junction_table.append(junction_pd[i].split())

pd_junction=pd.DataFrame(junction_table, columns=['Name', 'Elevation', 'MaxDepth', 'InitDepth', 'SurDepth', 'Aponded'])

# Merge df1 and df2 based on the 'Name' column
pd_junction = pd_junction.merge(coord_j, on='Name', how='left')

# Display the merged DataFrame
pd_junction

,Name,Elevation,MaxDepth,InitDepth,SurDepth,Aponded,X-Coord,Y-Coord
0,101,533.74,2.61,0.06,0,0,699585.924,248934.475
1,102a,533.46,3.08,0.04,0,0,699525.484,248908.265
2,102b,531.98,4.12,0.05,0,0,699470.464,248886.535
3,102c,531.84,3.81,0.05,0,0,699450.184,248905.615
4,102d,533.27,3.11,0.01,0,0,699579.644,248927.845
...,...,...,...,...,...,...,...,...
422,SK_102_Steuerschacht,530.93,2.62,0.06,0,0,699326.194,248921.655
423,SK102.2,531.66,3.95,0.01,0,0,699439.784,248905.455
424,SK102_SBW,531.68,3.95,0.05,0,0,699439.974,248907.385
425,SS8,527.03,2,0.01,0,0,698966.373,249408.684


## BGI fix parameters

In the next cell, all the fix BGI parameters are initialized.

In [13]:
bgi_parameters=pd.DataFrame()

#LID
bgi_parameters["bgi_number"]= ["1"]
bgi_parameters["ini_sat"]= ["20"]
bgi_parameters["rpt_file"]= ["*"]
bgi_parameters["drain_to"]= ["*"]
#bgi_parameters["from_perv"]= ["0"] assigned below
  
#STORAGE fixed parameters
bgi_parameters["pond_maxdepth"]= ["2"] #same as biocell depth/between 1.2 and 2 m (woods-ballard et al. 2015)
bgi_parameters["pond_initdepth"]= ["0"]
bgi_parameters["pond_shape"]= ["FUNCTIONAL"] #give the parameters
bgi_parameters["pond_surcharge"]= ["2"] #heights when the pond causes flooding
bgi_parameters["pond_evap"]= ["1"] #max evaporation
bgi_parameters["pond_suct_head"] = ["0"] #mm suction head, to check the value
bgi_parameters["pond_infiltration"] = ["0"] # seepage mm/hr
bgi_parameters["pond_initdef"] = ["0"] #initial deficit, 0 if seepage is constant

#CONDUITS fixed parameters
#bgi_parameters["pipe_length"]= [""] #computed below
bgi_parameters["pipe_roughness"]= ["0.0117647061"] #taken from calibrated model
bgi_parameters["pipe_in_offset"]= ["1.5"] 
bgi_parameters["pipe_out_offset"]= ["0.1"] 
bgi_parameters["pipe_init_flow"]= ["0"]
bgi_parameters["pipe_max_flow"]= ["0"]

#XSECTIONS fixed parameters
bgi_parameters["pipe_shape"]= ["CIRCULAR"]
bgi_parameters["pipe_geom1"]= ["0.07"] 
bgi_parameters["pipe_geom2"]= ["0"] #equal to 0 if geometry is circular
bgi_parameters["pipe_geom3"]= ["0"]
bgi_parameters["pipe_geom4"]= ["0"]
bgi_parameters["pipe_barrels"]= ["1"] #number of barrels in conduit

#WEIRS fixed parameters
bgi_parameters["weir_type"]= "V-NOTCH"
#bgi_parameters["weir_crest_ht"]= "2" #height of the pond
bgi_parameters["weir_coef"] = "1.4" #SWMM values for v-notch
bgi_parameters["weir_gated"] = "NO"
bgi_parameters["weir_endcon"] = "0"
bgi_parameters["weir_endcoeff"] = "0"
bgi_parameters["weir_surcharge"] = "NO" #water level usually lower then the opening of the weir
bgi_parameters["weir_shape"] = "TRIANGULAR"

#dimensioning the diameteter of the pipe and weir
bgi_parameters["g_const"] = 9.81
bgi_parameters["cd_const"] = 0.6 #source: Design, operation and adaptation of reservoirs for flood storage
bgi_parameters["max_head"] = 0.5
bgi_parameters["weir_theta"] = np.pi/2 #source: Design, operation and adaptation of reservoirs for flood storage
bgi_parameters["rainfall_2y"] = rainfall_2y
bgi_parameters["rainfall_100y"] = rainfall_100y


bgi_parameters

,bgi_number,ini_sat,rpt_file,drain_to,pond_maxdepth,pond_initdepth,pond_shape,pond_surcharge,pond_evap,pond_suct_head,...,weir_endcon,weir_endcoeff,weir_surcharge,weir_shape,g_const,cd_const,max_head,weir_theta,rainfall_2y,rainfall_100y
0,1,20,*,*,2,0,FUNCTIONAL,2,1,0,...,0,0,NO,TRIANGULAR,9.81,0.6,0.5,1.570796,17.3,56.5


## Function that adds the ponds in the input file

In [14]:
#input files: array with ponds surface, subcatchment parameters, LIDs fixed parameters, text file of the existing subcatchment in the input file
#output fles: text files to be added in the swmm .inp file

def add_ponds(pond_surface,sub_par,lid_par,pd_sub,coord_j): 
    pond_string=[]
    coor_string=[]
    pipe_string=[]
    sect_string=[]
    subc_string=[]
    weir_string=[]
    
    pd_sub_pond = pd_sub.copy()
    
    #STORAGE fixed parameters
    
    pond_maxdepth= lid_par["pond_maxdepth"].to_string(index=False) #same as biocell
    pond_initdepth= lid_par["pond_initdepth"].to_string(index=False)
    pond_shape= lid_par["pond_shape"].to_string(index=False) #give the parameters
    pond_surcharge = lid_par["pond_surcharge"].to_string(index=False)
    pond_evap = lid_par["pond_evap"].to_string(index=False)
    pond_suct_head = bgi_parameters["pond_suct_head"].to_string(index=False)
    pond_infiltration = bgi_parameters["pond_infiltration"].to_string(index=False)
    pond_initdef = bgi_parameters["pond_initdef"].to_string(index=False)
    
    #CONDUITS fixed parameters
    
    #pipe_length = lid_par["pipe_length"].to_string(index=False) 
    pipe_roughness = lid_par["pipe_roughness"].to_string(index=False) #taken from calibrated model
    pipe_in_offset = lid_par["pipe_in_offset"].to_string(index=False) 
    pipe_out_offset = lid_par["pipe_out_offset"].to_string(index=False)
    pipe_init_flow = lid_par["pipe_init_flow"].to_string(index=False)
    pipe_max_flow = lid_par["pipe_max_flow"].to_string(index=False)
    
    #XSECTIONS fixed parameters
    
    pipe_shape = lid_par["pipe_shape"].to_string(index=False)
    pipe_geom1 = lid_par["pipe_geom1"].to_string(index=False) #diameter, given
    pipe_geom2 = lid_par["pipe_geom2"].to_string(index=False)
    pipe_geom3 = lid_par["pipe_geom3"].to_string(index=False) 
    pipe_geom4 = lid_par["pipe_geom4"].to_string(index=False)
    pipe_barrels= lid_par["pipe_barrels"].to_string(index=False)
    
    #WEIRS fixed parameters
    
    weir_type = lid_par["weir_type"].to_string(index=False)
    #weir_crest_ht = lid_par["weir_crest_ht"].to_string(index=False)
    weir_coef = lid_par["weir_coef"].to_string(index=False)#m3/s TO DECIDE, depending on the subcatchment?
    weir_gated = lid_par["weir_gated"].to_string(index=False)
    weir_endcon = lid_par["weir_endcon"].to_string(index=False)
    weir_endcoeff = lid_par["weir_endcoeff"].to_string(index=False)
    weir_surcharge = lid_par["weir_surcharge"].to_string(index=False)
    weir_shape = lid_par["weir_shape"].to_string(index=False)

    
    for j in range(len(pond_surface.index)): 
        if pond_surface[j]>0:
            sub_name = str(tot_surface.index[j])
            
            #STORAGE variables
            pond_name= "pond_" + sub_name
            pond_elev= str(sub_par["Surface Elevation (m)"][j]-1) #surface elevation of the subcatchment -1m

            a_value= str(pond_surface[j]/2.625) #this parameters enable to represend a trapezoidal pond with slope 1:3
            b_value= "1.4" #defined
            c_value= "0"
                        
            #COORDINATES variables
            pond_x= str(sub_par["X-Coordinate"][j]+10)  #x coordinate next to the subcatchment outlet
            pond_y= str(sub_par["Y-Coordinate"][j]+10)  #y coordinate next to the subcatchment outlet
            
            #CONDUITS variables
            pipe_name = pond_name + "-" + str(pd_sub["Outlet"][j])
            from_node = pond_name
            to_node = str(pd_sub["Outlet"][j])
            
            #compute pipe length (it could be computed outside the for loop, the length is always the same)
            for n in range(len(coord_j)):
                if coord_j["Name"][n] == str(pd_sub["Outlet"][j]):
                    x_node = coord_j["X-Coord"][n]
                    y_node = coord_j["Y-Coord"][n]
                
            # Convert Swiss coordinates to geographic coordinates
            lat1, lon1 = transformer.transform(sub_par["X-Coordinate"][j], sub_par["Y-Coordinate"][j]) #coordinates of the subcatchment
            lat2, lon2 = transformer.transform(x_node, y_node) #coordinates of the outlet node
            dlat = math.radians(lat2 - lat1)
            dlon = math.radians(lon2 - lon1)
            a = math.sin(dlat/2)**2 + math.cos(math.radians(lat1)) * math.cos(math.radians(lat2)) * math.sin(dlon/2)**2
            c = 2 * math.atan2(math.sqrt(a), math.sqrt(1-a))
            pipe_length = str(round(6371 * c *1000,2)) #
                 
            outflow_weir= bgi_parameters["rainfall_100y"][0]/(1000*3600)*pd_sub["Drainage Area (m2)"][j] #m3/s
            max_weir_h = (outflow_weir/(8/15*bgi_parameters["cd_const"][0]*(2*bgi_parameters["g_const"][0])**0.5*np.tan(bgi_parameters["weir_theta"][0]/2)))**(2/5)
            
            #required for the triangular geometry: Full Height, Top Width (Table D1, SWMM User Manual)
            if max_weir_h < 0.1:
                max_weir_h = 0.1
                
            weir_height = str(round(max_weir_h,2))
            weir_width = str(round(max_weir_h*2,2)) #since theta is 90°, the width of the triangle is 2*h
            
            weir_crest_ht = str(round(1.95-max_weir_h,2)) 
            
            #SUBCATCHMENT variables
            for k in range(len(pd_sub_pond)):
                if pd_sub_pond.iloc[k,0]==sub_name:
                    pd_sub_pond.iloc[k,2]=pond_name
            
            #weir variables
            weir_name = "weir_" + pond_name
            weir_from_node = pond_name
            weir_to_node = str(pd_sub["Outlet"][j])
            

            pond_string.append(pond_name +" "+ pond_elev+" "+ pond_maxdepth +" "+ pond_initdepth +" "+ pond_shape +" "+ a_value +" "+ b_value +" "+ c_value +" "+ pond_surcharge +" "+pond_evap +" "+ pond_suct_head +" "+ pond_infiltration +" "+ pond_initdef +" \n")
            coor_string.append(pond_name +" "+ pond_x+" "+ pond_y+" \n")
            pipe_string.append(pipe_name +" "+ from_node +" "+ to_node +" "+ pipe_length +" "+ pipe_roughness +" "+ pipe_in_offset +" "+ pipe_out_offset +" "+ pipe_init_flow +" "+ pipe_max_flow +" \n")
            sect_string.append(pipe_name +" "+ pipe_shape +" "+ pipe_geom1 +" "+ pipe_geom2 +" "+ pipe_geom3 +" "+ pipe_geom4 +" "+ pipe_barrels +" \n")
            sect_string.append(weir_name +" "+ weir_shape +" "+ weir_height +" "+ weir_width +" "+ pipe_geom3 +" "+ pipe_geom4 +" "+ pipe_barrels +" \n")
            weir_string.append(weir_name +" "+ weir_from_node +" "+ weir_to_node +" "+ weir_type +" "+ weir_crest_ht +" "+ weir_coef +" "+ weir_gated+" "+weir_endcon +" "+ weir_endcoeff +" "+ weir_surcharge +" \n")
    
            

    pond_string.append("\n")
    coor_string.append("\n")
    pipe_string.append("\n")
    sect_string.append("\n")
    weir_string.append("\n")
    
            
    new_pond_text=pond_string
    new_coor_text=coor_string
    new_pipe_text=pipe_string
    new_sect_text=sect_string
    new_weir_text=weir_string
    
    
    return new_pond_text,new_coor_text,new_pipe_text,new_sect_text,new_weir_text,pd_sub_pond 

## Function that generates input files including ponds

To change the routing assumption:
- i) no routing: for BC --> from_imp = "0" and to_perv = "0"
- ii) impervious: for BC --> from_imp = "100" and to_perv = "0"
- iii) from impervious and BGI: run the function "write_input_file_assumption_iii"  instead of "write_input_file"
- iv) from impervious and pervious: for BC --> from_imp = "100" and to_perv = "100"

In [29]:
routing_scheme = pd.DataFrame({
    "i": [0, 0, False],
    "ii": [100, 0, False],
    "iii": [100, 0, True],
    "iv": [100, 100, False]
}, index=["from_imp", "from_perv", "from_bgi"])

routing_scheme

,i,ii,iii,iv
from_imp,0,100,100,100
from_perv,0,0,0,100
from_bgi,False,False,True,False


In [30]:
#input: libreries with the scenarios, total bgi area, position where the LID section of the .inp file starts and ends, parameters of the sub-catchments, bgi parameters, coordinates of the sub-catchments, name of the file, and boolean required to save the generated files
#output: the function prints the .inp files with the bgi scenarios (if write = True)

def write_input_file(model,tot_surface,start_LID,end_LID,sub_par,pd_sub,lid_par,coord_j,file_name,write,rout):
        
    #fix parameters
    bgi_number = lid_par["bgi_number"].to_string(index=False)
    ini_sat = lid_par["ini_sat"].to_string(index=False)
    rpt_file = lid_par["rpt_file"].to_string(index=False)
    drain_to = lid_par["drain_to"].to_string(index=False)
    
    #change imperviousness
    pd_sub_area=pd_sub.iloc[:,3].astype(float)
    pd_sub_imp=pd_sub.iloc[:,4].astype(float)
    sum_pp_gr=model["PP"]+model["GR"]

    perv_area=pd_sub_area*(1-pd_sub_imp/100)
    new_imp=[]
    for i in range(len(pd_sub_area)): #compute the imperviousness (updated in case PP and GR are in the scenario)
        temp_imp=100-((sum_pp_gr[i]/10000+perv_area[i])/pd_sub_area[i])*100
        if temp_imp<0:
            print("Imperviousness = " + str(temp_imp) + " changed to 0") #print this sentence if imperviousness was smaller than 0 (due to areas approximations)
            temp_imp=0
        temp_imp = round(temp_imp,1)
        new_imp.append(temp_imp)
    
    bgi_string=[]
    for i in tot_surface:

        #parameters that depend on the BGI type
        bgi = i
        if bgi == "BC":
            if sum(model[bgi])>0:
                bgi_type = "bcell"
                from_imp = str(routing_scheme[rout]["from_imp"])
                from_perv = str(routing_scheme[rout]["from_perv"])
                to_perv = "0"
                
        if bgi == "PP":
            if sum(model[bgi])>0:
                bgi_type = "ppave"
                from_imp = "0"
                to_perv = "0"
                from_perv = "0"

        if bgi == "GR":
            if sum(model[bgi])>0:
                bgi_type = "groof"
                from_imp = "0"
                to_perv = "0"
                from_perv = "0"
                
        if bgi == "UP":
            #add function for ponds
            if sum(model[bgi])>0:
                [pond_txt, coor_txt, pipe_txt, sect_txt, weir_txt, pd_sub_lid] = add_ponds(model[bgi],sub_par,lid_par,pd_sub,coord_j)
            break
            
        # Modify the LID control section
        for j in range(len(tot_surface.index)):
            if model[bgi][j]>0:
                name = str(tot_surface.index[j])
                bgi_area = model[bgi][j]
                bgi_width= bgi_area**0.5
                
                if routing_scheme[rout]["from_bgi"] == True: #routing assumption where the runoff of PP and GR is routed to BC
                    #if there are both PP or GR and BC, runoff is routed to BC
                    if bgi == "BC" and (model["PP"][j]>0 or model["GR"][j]>0):
                        from_perv = (((model["PP"][j]+model["GR"][j])/10000)/((1-new_imp[j]/100)*pd_sub_area[j]))*100 #compute the percentage of the sub-catchment that PP and GR cover

                        if from_perv < 0 or from_perv > 100: #check that the percentage is between 0% and 100%
                            raise ValueError("Variable must be between 0 and 100")
                    else:
                        from_perv = 0 #if not on BC with also PP/GR, set it to 0
                    
                bgi_string.append(name +" "+ bgi_type+" "+ bgi_number+" "+str(bgi_area)+" "+str(bgi_width)+" "+ini_sat+" "+from_imp+" "+to_perv+" "+rpt_file+" "+drain_to+" "+str(from_perv) +" \n")
                                        
                
        bgi_string.append("\n")    #add an empty line for the last BGI
    
    subc_string=[]  #initialize array where the info of the subcatchment will be saved
    
    
    for k in range(len(pd_sub)):
        if sum(model["UP"])>0:
            subc_string.append(str(pd_sub_lid.iloc[k,0])+" "+str(pd_sub_lid.iloc[k,1])+" "+str(pd_sub_lid.iloc[k,2])+" "+str(pd_sub_lid.iloc[k,3])+" "+str(new_imp[k])+" "+str(pd_sub_lid.iloc[k,5])+" "+str(pd_sub_lid.iloc[k,6])+" "+str(pd_sub_lid.iloc[k,7])+" \n")
        else:
            subc_string.append(str(pd_sub.iloc[k,0])+" "+str(pd_sub.iloc[k,1])+" "+str(pd_sub.iloc[k,2])+" "+str(pd_sub.iloc[k,3])+" "+str(new_imp[k])+" "+str(pd_sub.iloc[k,5])+" "+str(pd_sub.iloc[k,6])+" "+str(pd_sub.iloc[k,7])+" \n")
        
    subc_string.append("\n")
    
    
    if sum(model["UP"])<=0:
        new_text_file=lines_input[:start_sub+2]+subc_string+lines_input[end_sub-1:start_LID+2]+bgi_string+lines_input[end_LID-1:]
    else:
        new_text_file = lines_input[:start_sub+2]+subc_string+lines_input[end_sub-1:start_LID+2]+bgi_string+lines_input[end_LID-1:end_storage-2]+pond_txt+lines_input[end_storage-1:end_conduits-2]+pipe_txt+lines_input[end_conduits-1:end_weirs-2]+weir_txt+lines_input[end_weirs-1:end_xsections-2]+sect_txt+lines_input[end_xsections-1:end_coordinates-2]+coor_txt+lines_input[end_coordinates-1:]


    if write==True:
        # Write the modified contents back to the file
        with open('swmm_files/generated_scenarios/' + file_name + '.inp', 'w') as f:
            f.writelines(new_text_file)
            print(file_name + " was created")

        f.close()

Run the functions and save the swmm files

In [31]:
routing = "i" #set the routing assumption
writeT=True #if true, the swmm files will be saved

for i in lib_scenarios.keys():
    outlet = write_input_file(lib_scenarios[i],tot_surface,start_LID,end_LID,sub_parameters,pd_sub,bgi_parameters,coord_j,i,writeT,routing)

sc_0_0.25_BC was created
sc_0_0.5_BC was created
sc_0_0.75_BC was created
sc_0_1_BC was created
sc_1_0.25_PP was created
sc_1_0.5_PP was created
sc_1_0.75_PP was created
Imperviousness = -0.02688844693203407 changed to 0
Imperviousness = -0.09004883272727682 changed to 0
sc_1_1_PP was created
sc_2_0.25_GR was created
sc_2_0.5_GR was created
sc_2_0.75_GR was created
sc_2_1_GR was created
sc_3_0.25_UP was created
sc_3_0.5_UP was created
sc_3_0.75_UP was created
sc_3_1_UP was created
sc_4_0.25_BC_PP was created
sc_4_0.5_BC_PP was created
sc_4_0.75_BC_PP was created
Imperviousness = -0.02688844693203407 changed to 0
Imperviousness = -0.09004883272727682 changed to 0
sc_4_1_BC_PP was created
sc_5_0.25_PP_GR was created
sc_5_0.5_PP_GR was created
sc_5_0.75_PP_GR was created
Imperviousness = -0.02688844693203407 changed to 0
Imperviousness = -0.09004883272727682 changed to 0
sc_5_1_PP_GR was created
sc_6_0.25_BC_GR was created
sc_6_0.5_BC_GR was created
sc_6_0.75_BC_GR was created
sc_6_1_BC_G

## Statistics about BGI areas

Compute the actual BGI area implemented for each scenario

In [18]:
tot_bgi_area_scenarios=[]

for i in lib_scenarios.keys():
    tot_bgi_area_scenarios.append(np.array(np.sum(lib_scenarios[i])))

    tot_bgi_area = np.sum(np.sum(lib_scenarios[i]))
    

reshaped_data = np.array(tot_bgi_area_scenarios).reshape((60, 4))
    
tot_bgi_area_df = pd.DataFrame()

tot_bgi_area_df["BC"]=reshaped_data[:,0]
tot_bgi_area_df["PP"]=reshaped_data[:,1]
tot_bgi_area_df["GR"]=reshaped_data[:,2]
tot_bgi_area_df["DP"]=reshaped_data[:,3]

#tot_bgi_area_df.to_csv("tot_bgi_area.csv", index=False)
tot_bgi_area_df

,BC,PP,GR,DP
0,87506.307750,0.000000,0.000000,0.000000
1,175012.615500,0.000000,0.000000,0.000000
2,262524.701250,0.000000,0.000000,0.000000
3,350032.935000,0.000000,0.000000,0.000000
4,0.000000,57457.456276,0.000000,0.000000
5,0.000000,114921.740024,0.000000,0.000000
6,0.000000,172382.610036,0.000000,0.000000
7,0.000000,229843.480048,0.000000,0.000000
8,0.000000,0.000000,25722.075479,0.000000
9,0.000000,0.000000,51509.361755,0.000000


Show statistics about the BGI areas in the different sub-catchments

In [19]:
def sc_metrics(sc):
    print("Total area: " + str(np.sum(sc)) + "m2")
    count = np.sum(sc > 0)
    print("Subcatchment number: " + str(count))
    print("Average Surface: " + str(np.average(sc[sc> 0]))+ "m2")
    print("Median surface: " + str(np.median(sc[sc> 0]))+ "m2")
    
#sc_metrics(lib_scenarios["sc_0_1_BC"]["BC"])
sc_metrics(lib_scenarios["sc_1_1_PP"]["PP"])
#sc_metrics(lib_scenarios["sc_2_1_GR"]["GR"])
#sc_metrics(lib_scenarios["sc_3_1_UP"]["UP"])

Total area: 229843.480047865m2
Subcatchment number: 238
Average Surface: 965.7289077641386m2
Median surface: 585.7175549m2
